# Advanced Problems with Solutions: `yield from` and Two-Way Generator Communication

This notebook develops a protocol-level understanding of Python generator delegation. It goes beyond simple iteration and exercises all major communication paths between a caller, a delegating generator, and a subgenerator:

- normal iteration with `next()`
- values sent with `.send(...)`
- exceptions injected with `.throw(...)`
- cancellation and cleanup with `.close()`
- subgenerator return values carried by `StopIteration.value`
- delegation-chain introspection
- recursive and streaming applications
- a manual implementation of the essential `yield from` protocol

Every problem includes a complete solution and executable verification. All code uses only the Python standard library.


## How to use this notebook

1. Read each problem before opening the solution mentally.
2. Predict the next yielded value, state transition, or final return value.
3. Run the solution cell.
4. Run the verification cell. Assertions should pass without output.
5. Modify the examples and observe how `next`, `send`, `throw`, and `close` travel through the delegation chain.

### Terminology

- **Caller**: code that drives a generator with `next`, `send`, `throw`, or `close`.
- **Delegator**: a generator containing `yield from subgenerator`.
- **Subgenerator**: the object receiving delegated operations.
- **Yielded value**: travels from subgenerator → delegator → caller.
- **Sent value / thrown exception / close request**: travels from caller → delegator → subgenerator.


## Best-practice checklist

- Prime a coroutine-style generator with `next(gen)` before sending a non-`None` value.
- Use a unique sentinel object rather than a magic string such as `"STOP"`.
- Document the generator's **yield type**, **send type**, and **return type**.
- Put resource cleanup in `finally` so `.close()` and exceptions cannot bypass it.
- Catch only exceptions the generator can meaningfully handle.
- Use `yield from` when you want transparent forwarding of `next`, `send`, `throw`, `close`, and the subgenerator's return value.
- Keep tests deterministic and explicitly verify final `StopIteration.value` when it matters.
- Avoid depending on implementation details such as local-variable names in production code; introspection is most suitable for teaching and debugging.


In [1]:
from __future__ import annotations

from collections.abc import Generator, Iterable, Iterator, Mapping
from dataclasses import dataclass
from inspect import getgeneratorlocals, getgeneratorstate, isgenerator
from itertools import islice
from typing import Any, TypeVar
import inspect
import math
import sys

T = TypeVar("T")
STOP = object()  # unique protocol sentinel


## Utility: drain a generator and capture its return value

A `for` loop discards the value attached to `StopIteration`. The helper below captures both ordinary yielded values and the generator's final return value.


In [2]:
def drain(generator: Generator[T, None, Any]) -> tuple[list[T], Any]:
    """Consume a generator and return (yielded_values, return_value)."""
    yielded: list[T] = []
    while True:
        try:
            yielded.append(next(generator))
        except StopIteration as exc:
            return yielded, exc.value


## Mini-example A — Manual iteration forwarding versus `yield from`

These two delegators produce the same ordinary values. The important difference appears when the caller uses `send`, `throw`, or `close`, or when the subgenerator returns a value.


In [3]:
def squares(limit: int) -> Generator[int, None, None]:
    for number in range(limit):
        yield number * number


def manual_value_forwarder(limit: int) -> Generator[int, None, None]:
    for value in squares(limit):
        yield value


def native_delegator(limit: int) -> Generator[int, None, None]:
    yield from squares(limit)


assert list(manual_value_forwarder(6)) == [0, 1, 4, 9, 16, 25]
assert list(native_delegator(6)) == [0, 1, 4, 9, 16, 25]


## Mini-example B — A subgenerator's `return` becomes `StopIteration.value`

Inside a delegator, the expression `result = yield from subgenerator()` evaluates to the subgenerator's return value.


In [4]:
def three_values() -> Generator[int, None, str]:
    yield 10
    yield 20
    yield 30
    return "subgenerator complete"


def return_value_demo() -> Generator[int | str, None, str]:
    result = yield from three_values()
    yield f"captured: {result}"
    return result


values, final_value = drain(return_value_demo())
assert values == [10, 20, 30, "captured: subgenerator complete"]
assert final_value == "subgenerator complete"


# Problem 1 — Audit a delegated stream and preserve its return value

Implement a subgenerator that:

1. validates that every item is a non-negative integer (reject booleans),
2. yields each valid item unchanged,
3. returns a dictionary containing the item count and sum.

Then implement a delegator that uses `yield from`, emits one final human-readable audit line, and returns the same dictionary.

**Protocol:** yields `int | str`, receives nothing, returns `dict[str, int]`.


### Solution 1

The subgenerator owns validation and statistics. The delegator remains small: it captures the result of `yield from`, adds one outward-facing message, and returns the captured result.


In [5]:
def validated_numbers(
    values: Iterable[int],
) -> Generator[int, None, dict[str, int]]:
    count = 0
    total = 0

    for value in values:
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"expected int, received {type(value).__name__}")
        if value < 0:
            raise ValueError(f"negative value is not allowed: {value}")

        count += 1
        total += value
        yield value

    return {"count": count, "sum": total}


def audited_numbers(
    values: Iterable[int],
) -> Generator[int | str, None, dict[str, int]]:
    audit = yield from validated_numbers(values)
    yield f"audit complete: count={audit['count']}, sum={audit['sum']}"
    return audit


In [6]:
yielded, returned = drain(audited_numbers([2, 4, 6, 8]))

assert yielded == [
    2,
    4,
    6,
    8,
    "audit complete: count=4, sum=20",
]
assert returned == {"count": 4, "sum": 20}

try:
    list(audited_numbers([1, True, 3]))
except TypeError as exc:
    assert "bool" in str(exc)
else:
    raise AssertionError("a boolean should have been rejected")


# Problem 2 — Build a bidirectional running-statistics coroutine

Create a subgenerator that:

- first yields an empty statistics snapshot,
- accepts numeric samples through `.send(sample)`,
- yields an updated immutable snapshot after every sample,
- returns the final snapshot when it receives the unique `STOP` sentinel.

A delegator should use `yield from`, yield one summary line, and return the final snapshot.

**Protocol:** yields `StatisticsSnapshot | str`, receives `int | float | object`, returns `StatisticsSnapshot`.


### Solution 2

The assignment target in `incoming = yield snapshot` receives the value supplied by the caller's next `.send(...)`. A plain `next(generator)` is equivalent to `.send(None)`.


In [7]:
@dataclass(frozen=True)
class StatisticsSnapshot:
    count: int
    total: float
    mean: float | None
    minimum: float | None
    maximum: float | None


def running_statistics() -> Generator[
    StatisticsSnapshot,
    int | float | object,
    StatisticsSnapshot,
]:
    count = 0
    total = 0.0
    minimum: float | None = None
    maximum: float | None = None

    snapshot = StatisticsSnapshot(0, 0.0, None, None, None)

    while True:
        incoming = yield snapshot

        if incoming is STOP:
            return snapshot

        if isinstance(incoming, bool) or not isinstance(incoming, (int, float)):
            raise TypeError("samples must be int or float")

        sample = float(incoming)
        if not math.isfinite(sample):
            raise ValueError("samples must be finite")

        count += 1
        total += sample
        minimum = sample if minimum is None else min(minimum, sample)
        maximum = sample if maximum is None else max(maximum, sample)
        snapshot = StatisticsSnapshot(
            count=count,
            total=total,
            mean=total / count,
            minimum=minimum,
            maximum=maximum,
        )


def statistics_session(
    name: str,
) -> Generator[StatisticsSnapshot | str, int | float | object, StatisticsSnapshot]:
    final_snapshot = yield from running_statistics()
    yield f"{name}: {final_snapshot.count} samples, mean={final_snapshot.mean}"
    return final_snapshot


In [8]:
session = statistics_session("sensor-A")

empty = next(session)  # prime the delegation chain
one = session.send(10)
two = session.send(20)
three = session.send(-5)
summary_line = session.send(STOP)

assert empty == StatisticsSnapshot(0, 0.0, None, None, None)
assert one.mean == 10.0
assert two.mean == 15.0
assert three == StatisticsSnapshot(3, 25.0, 25.0 / 3, -5.0, 20.0)
assert summary_line.startswith("sensor-A: 3 samples")

try:
    next(session)
except StopIteration as exc:
    assert exc.value == three
else:
    raise AssertionError("the session should be complete")


# Problem 3 — Forward exceptions with `.throw(...)`

Design a sampler that accepts values with `.send(...)` and supports two caller-injected control exceptions:

- `SkipSample`: skip one position without adding a value.
- `AbortSampling(reason)`: stop immediately and return a report.

Wrap it in a delegator. Do not manually forward exceptions; rely on `yield from`.


### Solution 3

When the delegator is suspended at `yield from`, `delegator.throw(...)` is forwarded to the subgenerator's `.throw(...)`. The exception is raised at the subgenerator's current suspension point.


In [9]:
class SkipSample(Exception):
    """Tell the sampler to record a skipped position."""


class AbortSampling(Exception):
    """Tell the sampler to stop and record a reason."""


@dataclass(frozen=True)
class SamplingReport:
    accepted: tuple[Any, ...]
    skipped: int
    reason: str


def resilient_sampler() -> Generator[str, Any, SamplingReport]:
    accepted: list[Any] = []
    skipped = 0

    while True:
        try:
            value = yield f"ready: accepted={len(accepted)}, skipped={skipped}"
        except SkipSample:
            skipped += 1
            continue
        except AbortSampling as exc:
            return SamplingReport(tuple(accepted), skipped, str(exc))
        else:
            accepted.append(value)


def delegated_sampler() -> Generator[str | SamplingReport, Any, SamplingReport]:
    report = yield from resilient_sampler()
    yield report
    return report


In [10]:
sampler = delegated_sampler()

assert next(sampler) == "ready: accepted=0, skipped=0"
assert sampler.send(11) == "ready: accepted=1, skipped=0"
assert sampler.throw(SkipSample()) == "ready: accepted=1, skipped=1"
assert sampler.send(29) == "ready: accepted=2, skipped=1"

report_yielded = sampler.throw(AbortSampling("operator requested stop"))
assert report_yielded == SamplingReport((11, 29), 1, "operator requested stop")

try:
    next(sampler)
except StopIteration as exc:
    assert exc.value == report_yielded
else:
    raise AssertionError("the delegated sampler should be complete")


# Problem 4 — Prove that `.close()` propagates and cleanup runs

Create a three-level delegation chain:

`outer_resource_stream → middle_resource_stream → resource_stream`

The innermost generator must append `"open:<name>"` to a log when it starts and `"close:<name>"` in a `finally` block. After yielding one value, close only the outer generator. Verify that all generators close and cleanup runs exactly once.


### Solution 4

Closing the delegator raises `GeneratorExit` at its suspension point. `yield from` forwards `.close()` to the active subgenerator, and this repeats through the chain.


In [11]:
def resource_stream(name: str, log: list[str]) -> Generator[str, None, None]:
    log.append(f"open:{name}")
    try:
        index = 0
        while True:
            yield f"{name}:{index}"
            index += 1
    finally:
        log.append(f"close:{name}")


def middle_resource_stream(
    name: str,
    log: list[str],
) -> Generator[str, None, None]:
    yield from resource_stream(name, log)


def outer_resource_stream(
    name: str,
    log: list[str],
) -> Generator[str, None, None]:
    yield from middle_resource_stream(name, log)


In [12]:
cleanup_log: list[str] = []
outer = outer_resource_stream("database", cleanup_log)

assert next(outer) == "database:0"
middle = outer.gi_yieldfrom
inner = middle.gi_yieldfrom

assert getgeneratorstate(outer) == "GEN_SUSPENDED"
assert getgeneratorstate(middle) == "GEN_SUSPENDED"
assert getgeneratorstate(inner) == "GEN_SUSPENDED"

outer.close()

assert cleanup_log == ["open:database", "close:database"]
assert getgeneratorstate(outer) == "GEN_CLOSED"
assert getgeneratorstate(middle) == "GEN_CLOSED"
assert getgeneratorstate(inner) == "GEN_CLOSED"

# Closing again is harmless and must not duplicate cleanup.
outer.close()
assert cleanup_log == ["open:database", "close:database"]


# Problem 5 — Reimplement the essential `yield from` protocol manually

Write `manual_yield_from(iterable)` so that it forwards:

- `next()`
- `.send(value)`
- `.throw(exception)`
- `.close()`
- the subiterator's final `StopIteration.value`

Then compare it with native `yield from` using the same scripted endpoint.

> This problem is intentionally low-level. In application code, prefer native `yield from`.


### Solution 5

The implementation below mirrors the essential behavior specified by the generator delegation protocol. `GeneratorExit` is handled separately because a close request should invoke the child's `.close()` method before being re-raised.


In [13]:
def manual_yield_from(iterable: Iterable[T]) -> Generator[T, Any, Any]:
    iterator = iter(iterable)

    try:
        current = next(iterator)
    except StopIteration as exc:
        return exc.value

    while True:
        try:
            sent = yield current
        except GeneratorExit:
            close_method = getattr(iterator, "close", None)
            if close_method is not None:
                close_method()
            raise
        except BaseException:
            exception_info = sys.exc_info()
            throw_method = getattr(iterator, "throw", None)
            if throw_method is None:
                raise

            try:
                current = throw_method(*exception_info)
            except StopIteration as exc:
                return exc.value
            finally:
                del exception_info
        else:
            try:
                if sent is None:
                    current = next(iterator)
                else:
                    send_method = getattr(iterator, "send", None)
                    if send_method is None:
                        raise AttributeError(
                            f"{type(iterator).__name__} does not support send"
                        )
                    current = send_method(sent)
            except StopIteration as exc:
                return exc.value


In [14]:
class SoftReset(Exception):
    pass


def scripted_endpoint(log: list[str]) -> Generator[str | int, int | str, int]:
    total = 0
    log.append("started")

    try:
        command: int | str = yield "ready"
        while True:
            try:
                if command == "stop":
                    return total

                total += int(command)
                command = yield total
            except SoftReset:
                total = 0
                command = yield "reset"
    finally:
        log.append("closed")


def native_wrapper(log: list[str]) -> Generator[str | int, Any, int]:
    result = yield from scripted_endpoint(log)
    return result


def manual_wrapper(log: list[str]) -> Generator[str | int, Any, int]:
    result = yield from manual_yield_from(scripted_endpoint(log))
    return result


def drive_script(factory: Any) -> tuple[list[str | int], int, list[str]]:
    log: list[str] = []
    generator = factory(log)
    transcript: list[str | int] = [next(generator)]
    transcript.append(generator.send(3))
    transcript.append(generator.throw(SoftReset()))
    transcript.append(generator.send(4))

    try:
        generator.send("stop")
    except StopIteration as exc:
        returned = exc.value
    else:
        raise AssertionError("the scripted endpoint should have returned")

    return transcript, returned, log


native_result = drive_script(native_wrapper)
manual_result = drive_script(manual_wrapper)

assert native_result == manual_result
assert native_result == (["ready", 3, "reset", 4], 4, ["started", "closed"])

# Also verify close forwarding in the manual implementation.
manual_close_log: list[str] = []
manual_closer = manual_wrapper(manual_close_log)
assert next(manual_closer) == "ready"
manual_closer.close()
assert manual_close_log == ["started", "closed"]


C:\Users\user1\AppData\Local\Temp\ipykernel_5544\2299825337.py:24: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  current = throw_method(*exception_info)


# Problem 6 — Introspect an active delegation chain

Implement a debugger that walks through a generator's active `.gi_yieldfrom` chain and reports:

- depth,
- generator function name,
- generator state,
- visible local-variable names.

Use it on a three-level chain after the first value has been yielded.


### Solution 6

`.gi_yieldfrom` is either the currently delegated-to object or `None`. The chain is an implementation-level debugging aid, not a portable application interface for arbitrary iterators.


In [15]:
@dataclass(frozen=True)
class GeneratorFrameInfo:
    depth: int
    function_name: str
    state: str
    local_names: tuple[str, ...]


def describe_generator_chain(generator: Generator[Any, Any, Any]) -> list[GeneratorFrameInfo]:
    information: list[GeneratorFrameInfo] = []
    current: Any = generator
    depth = 0

    while isgenerator(current):
        information.append(
            GeneratorFrameInfo(
                depth=depth,
                function_name=current.gi_code.co_name,
                state=getgeneratorstate(current),
                local_names=tuple(sorted(getgeneratorlocals(current))),
            )
        )

        child = current.gi_yieldfrom
        if not isgenerator(child):
            break

        current = child
        depth += 1

    return information


def introspection_leaf() -> Generator[str, str, str]:
    leaf_local = "visible"
    response = yield "leaf-ready"
    return f"{leaf_local}:{response}"


def introspection_middle() -> Generator[str, str, str]:
    middle_local = 42
    result = yield from introspection_leaf()
    return f"middle({middle_local}):{result}"


def introspection_outer() -> Generator[str, str, str]:
    outer_local = True
    result = yield from introspection_middle()
    return f"outer({outer_local}):{result}"


In [16]:
chain = introspection_outer()
assert next(chain) == "leaf-ready"

frames = describe_generator_chain(chain)
assert [frame.function_name for frame in frames] == [
    "introspection_outer",
    "introspection_middle",
    "introspection_leaf",
]
assert all(frame.state == "GEN_SUSPENDED" for frame in frames)
assert "outer_local" in frames[0].local_names
assert "middle_local" in frames[1].local_names
assert "leaf_local" in frames[2].local_names

try:
    chain.send("done")
except StopIteration as exc:
    assert exc.value == "outer(True):middle(42):visible:done"
else:
    raise AssertionError("the chain should have returned")


# Problem 7 — Recursive tree traversal with cycle detection

Use recursive `yield from` to traverse a tree in either preorder or postorder. Yield `(path, label)` pairs, where `path` is a tuple of child indexes from the root.

Requirements:

- support `order="pre"` and `order="post"`,
- optionally stop descending beyond `max_depth`,
- detect cycles along the active recursion path,
- keep cleanup correct even if traversal raises.


### Solution 7

The `active` set tracks only the current recursion path. This allows a shared subtree to appear in two branches without being falsely classified as a cycle.


In [17]:
@dataclass
class TreeNode:
    label: str
    children: list["TreeNode"]


def walk_tree(
    root: TreeNode,
    *,
    order: str = "pre",
    max_depth: int | None = None,
) -> Generator[tuple[tuple[int, ...], str], None, None]:
    if order not in {"pre", "post"}:
        raise ValueError("order must be 'pre' or 'post'")
    if max_depth is not None and max_depth < 0:
        raise ValueError("max_depth must be non-negative or None")

    active: set[int] = set()

    def visit(
        node: TreeNode,
        path: tuple[int, ...],
        depth: int,
    ) -> Generator[tuple[tuple[int, ...], str], None, None]:
        if max_depth is not None and depth > max_depth:
            return

        marker = id(node)
        if marker in active:
            raise ValueError(f"cycle detected at path {path}")

        active.add(marker)
        try:
            if order == "pre":
                yield path, node.label

            for index, child in enumerate(node.children):
                yield from visit(child, path + (index,), depth + 1)

            if order == "post":
                yield path, node.label
        finally:
            active.remove(marker)

    yield from visit(root, (), 0)


In [18]:
tree = TreeNode(
    "root",
    [
        TreeNode("left", [TreeNode("left.left", [])]),
        TreeNode("right", [TreeNode("right.left", []), TreeNode("right.right", [])]),
    ],
)

assert list(walk_tree(tree, order="pre")) == [
    ((), "root"),
    ((0,), "left"),
    ((0, 0), "left.left"),
    ((1,), "right"),
    ((1, 0), "right.left"),
    ((1, 1), "right.right"),
]

assert [label for _, label in walk_tree(tree, order="post")] == [
    "left.left",
    "left",
    "right.left",
    "right.right",
    "right",
    "root",
]

assert [label for _, label in walk_tree(tree, max_depth=1)] == [
    "root",
    "left",
    "right",
]

cyclic = TreeNode("cycle-root", [])
cyclic.children.append(cyclic)
try:
    list(walk_tree(cyclic))
except ValueError as exc:
    assert "cycle detected" in str(exc)
else:
    raise AssertionError("the cycle should have been detected")


# Problem 8 — Safely flatten heterogeneous nested iterables

Implement a lazy recursive flattener using `yield from`.

Rules:

- strings and byte sequences are atomic,
- mappings are atomic,
- other iterables are expanded,
- `max_depth` can limit expansion,
- self-referential structures must raise a clear error.


### Solution 8

The algorithm delegates each nested iterable to the same recursive helper. As in the tree problem, active-path tracking prevents infinite recursion while still permitting shared objects in separate branches.


In [19]:
def flatten_nested(
    value: Any,
    *,
    max_depth: int | None = None,
) -> Generator[Any, None, None]:
    if max_depth is not None and max_depth < 0:
        raise ValueError("max_depth must be non-negative or None")

    active: set[int] = set()
    atomic_types = (str, bytes, bytearray)

    def is_expandable(item: Any) -> bool:
        return (
            isinstance(item, Iterable)
            and not isinstance(item, atomic_types)
            and not isinstance(item, Mapping)
        )

    def visit(item: Any, depth: int) -> Generator[Any, None, None]:
        if max_depth is not None and depth >= max_depth:
            yield item
            return

        if not is_expandable(item):
            yield item
            return

        marker = id(item)
        if marker in active:
            raise ValueError("cycle detected while flattening")

        active.add(marker)
        try:
            for child in item:
                yield from visit(child, depth + 1)
        finally:
            active.remove(marker)

    yield from visit(value, 0)


In [20]:
nested = [1, (2, [3, 4]), "abc", {"x": 5}, range(6, 8)]
assert list(flatten_nested(nested)) == [1, 2, 3, 4, "abc", {"x": 5}, 6, 7]

# At depth 1, the outer list is expanded, but its nested containers are atomic.
assert list(flatten_nested(nested, max_depth=1)) == [
    1,
    (2, [3, 4]),
    "abc",
    {"x": 5},
    range(6, 8),
]

self_reference: list[Any] = [1]
self_reference.append(self_reference)
try:
    list(flatten_nested(self_reference))
except ValueError as exc:
    assert "cycle detected" in str(exc)
else:
    raise AssertionError("the self-reference should have been rejected")


# Problem 9 — Stream text chunks into lines and return parser metrics

Build a subgenerator that accepts an iterable of text chunks, yields every complete newline-terminated line, and returns metrics containing:

- number of chunks,
- number of complete lines,
- number of characters,
- any trailing fragment.

Then build a delegator that uses `yield from`, emits the trailing fragment as a final line if present, and returns adjusted metrics.


### Solution 9

The subgenerator is deliberately strict about complete lines. The delegator changes the policy for the final fragment without duplicating the chunk-processing loop.


In [21]:
@dataclass(frozen=True)
class LineMetrics:
    chunks: int
    lines: int
    characters: int
    trailing_fragment: str


def complete_lines(chunks: Iterable[str]) -> Generator[str, None, LineMetrics]:
    chunks_seen = 0
    lines_seen = 0
    characters_seen = 0
    buffer = ""

    for chunk in chunks:
        if not isinstance(chunk, str):
            raise TypeError("all chunks must be strings")

        chunks_seen += 1
        characters_seen += len(chunk)
        buffer += chunk

        while "\n" in buffer:
            line, buffer = buffer.split("\n", 1)
            lines_seen += 1
            yield line

    return LineMetrics(
        chunks=chunks_seen,
        lines=lines_seen,
        characters=characters_seen,
        trailing_fragment=buffer,
    )


def all_lines(chunks: Iterable[str]) -> Generator[str, None, LineMetrics]:
    metrics = yield from complete_lines(chunks)

    if metrics.trailing_fragment:
        yield metrics.trailing_fragment
        metrics = LineMetrics(
            chunks=metrics.chunks,
            lines=metrics.lines + 1,
            characters=metrics.characters,
            trailing_fragment="",
        )

    return metrics


In [22]:
text_chunks = ["alpha\nb", "eta\ngamma", "\ndelta"]
lines, metrics = drain(all_lines(text_chunks))

assert lines == ["alpha", "beta", "gamma", "delta"]
assert metrics == LineMetrics(
    chunks=3,
    lines=4,
    characters=sum(map(len, text_chunks)),
    trailing_fragment="",
)


# Problem 10 — Adaptive batching controlled with `.send(...)`

Create a subgenerator that lazily consumes an iterable and yields batches. The caller may send a new positive batch size after each batch. Sending `None` keeps the current size. When the source is exhausted, return a report.

Wrap it with a delegator that yields the report once and then returns it.


### Solution 10

The value sent by the caller becomes the result of the suspended `yield batch` expression. Because `yield from` transparently forwards `.send(...)`, the delegator requires no custom routing logic.


In [23]:
@dataclass(frozen=True)
class BatchReport:
    items: int
    batches: int
    final_batch_size: int


def _validate_batch_size(size: int) -> int:
    if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
        raise ValueError("batch size must be a positive integer")
    return size


def adaptive_batches(
    values: Iterable[T],
    initial_size: int,
) -> Generator[list[T], int | None, BatchReport]:
    size = _validate_batch_size(initial_size)
    iterator = iter(values)
    item_count = 0
    batch_count = 0

    while True:
        batch = list(islice(iterator, size))
        if not batch:
            return BatchReport(item_count, batch_count, size)

        item_count += len(batch)
        batch_count += 1
        requested_size = yield batch

        if requested_size is not None:
            size = _validate_batch_size(requested_size)


def managed_batches(
    values: Iterable[T],
    initial_size: int,
) -> Generator[list[T] | BatchReport, int | None, BatchReport]:
    report = yield from adaptive_batches(values, initial_size)
    yield report
    return report


In [24]:
batches = managed_batches(range(10), initial_size=3)

assert next(batches) == [0, 1, 2]
assert batches.send(2) == [3, 4]
assert next(batches) == [5, 6]       # next(...) sends None
assert batches.send(4) == [7, 8, 9]

report = next(batches)
assert report == BatchReport(items=10, batches=4, final_batch_size=4)

try:
    next(batches)
except StopIteration as exc:
    assert exc.value == report
else:
    raise AssertionError("the batch manager should be complete")


# Problem 11 — Orchestrate multiple bidirectional worker sessions

Each worker should:

- yield an initial prompt,
- receive numeric values through `.send(...)`,
- yield an updated prompt after every value,
- return a report when it receives `STOP`.

The orchestrator should iterate through worker names and use `yield from` for each worker. A single `.send(STOP)` should finish the current worker and immediately return the next worker's first prompt. After the final worker, the orchestrator returns all reports.


### Solution 11

This demonstrates a powerful property: when one subgenerator returns, the delegator resumes immediately. It can start another `yield from` before the caller's method call returns.


In [25]:
@dataclass(frozen=True)
class WorkerPrompt:
    worker: str
    count: int
    subtotal: float


@dataclass(frozen=True)
class WorkerReport:
    worker: str
    count: int
    subtotal: float


def worker_session(
    name: str,
) -> Generator[WorkerPrompt, int | float | object, WorkerReport]:
    count = 0
    subtotal = 0.0
    prompt = WorkerPrompt(name, count, subtotal)

    while True:
        value = yield prompt

        if value is STOP:
            return WorkerReport(name, count, subtotal)

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("worker values must be numeric")

        count += 1
        subtotal += float(value)
        prompt = WorkerPrompt(name, count, subtotal)


def orchestrate_workers(
    names: Iterable[str],
) -> Generator[WorkerPrompt, int | float | object, list[WorkerReport]]:
    reports: list[WorkerReport] = []

    for name in names:
        report = yield from worker_session(name)
        reports.append(report)

    return reports


In [26]:
orchestrator = orchestrate_workers(["alpha", "beta"])

assert next(orchestrator) == WorkerPrompt("alpha", 0, 0.0)
assert orchestrator.send(5) == WorkerPrompt("alpha", 1, 5.0)
assert orchestrator.send(7) == WorkerPrompt("alpha", 2, 12.0)

# The same call ends alpha and starts beta.
assert orchestrator.send(STOP) == WorkerPrompt("beta", 0, 0.0)
assert orchestrator.send(3.5) == WorkerPrompt("beta", 1, 3.5)

try:
    orchestrator.send(STOP)
except StopIteration as exc:
    reports = exc.value
else:
    raise AssertionError("the orchestrator should be complete")

assert reports == [
    WorkerReport("alpha", 2, 12.0),
    WorkerReport("beta", 1, 3.5),
]


# Problem 12 — Capstone: a delegated form wizard with backtracking and cancellation

Create a multi-section wizard.

Each section subgenerator should:

- yield a prompt for its current field,
- accept a value with `.send(value)`,
- handle a caller-injected `Back` exception by moving to the previous field,
- handle `CancelWizard(reason)` by returning a cancelled section result.

The outer wizard should delegate to each section. It returns a complete `WizardResult`, or a cancelled result containing all information collected before cancellation.


### Solution 12

The caller uses normal values for data and exceptions for out-of-band control. `yield from` routes both channels to whichever section is currently active.


In [27]:
class Back(Exception):
    """Move to the previous field in the active section."""


class CancelWizard(Exception):
    """Cancel the active section and the entire wizard."""


@dataclass(frozen=True)
class FieldPrompt:
    section: str
    field: str
    index: int
    total: int
    existing_value: Any = None


@dataclass(frozen=True)
class SectionResult:
    section: str
    values: dict[str, Any]
    cancelled: bool
    reason: str | None = None


@dataclass(frozen=True)
class WizardResult:
    sections: dict[str, dict[str, Any]]
    cancelled: bool
    reason: str | None = None


def collect_section(
    section: str,
    fields: list[str],
) -> Generator[FieldPrompt, Any, SectionResult]:
    if not fields:
        return SectionResult(section, {}, False)

    values: dict[str, Any] = {}
    index = 0

    while index < len(fields):
        field = fields[index]
        prompt = FieldPrompt(
            section=section,
            field=field,
            index=index,
            total=len(fields),
            existing_value=values.get(field),
        )

        try:
            value = yield prompt
        except Back:
            if index > 0:
                index -= 1
                values.pop(fields[index], None)
            continue
        except CancelWizard as exc:
            return SectionResult(section, dict(values), True, str(exc))
        else:
            if value is None or value == "":
                # Reject empty values by asking for the same field again.
                continue
            values[field] = value
            index += 1

    return SectionResult(section, dict(values), False)


def form_wizard(
    schema: Mapping[str, list[str]],
) -> Generator[FieldPrompt, Any, WizardResult]:
    completed: dict[str, dict[str, Any]] = {}

    for section, fields in schema.items():
        result = yield from collect_section(section, fields)

        if result.cancelled:
            if result.values:
                completed[result.section] = result.values
            return WizardResult(completed, True, result.reason)

        completed[result.section] = result.values

    return WizardResult(completed, False, None)


In [28]:
schema = {
    "account": ["username", "email"],
    "profile": ["city"],
}

wizard = form_wizard(schema)
assert next(wizard) == FieldPrompt("account", "username", 0, 2, None)
assert wizard.send("alice") == FieldPrompt("account", "email", 1, 2, None)

# Backtracking removes the previous value and asks for it again.
assert wizard.throw(Back()) == FieldPrompt("account", "username", 0, 2, None)
assert wizard.send("alice_2") == FieldPrompt("account", "email", 1, 2, None)

# Completing a section immediately starts the next delegated section.
assert wizard.send("alice@example.test") == FieldPrompt("profile", "city", 0, 1, None)

try:
    wizard.send("Sofia")
except StopIteration as exc:
    completed_wizard = exc.value
else:
    raise AssertionError("the wizard should be complete")

assert completed_wizard == WizardResult(
    sections={
        "account": {
            "username": "alice_2",
            "email": "alice@example.test",
        },
        "profile": {"city": "Sofia"},
    },
    cancelled=False,
    reason=None,
)

# Cancellation example.
cancelled = form_wizard(schema)
next(cancelled)
cancelled.send("bob")
try:
    cancelled.throw(CancelWizard("user closed the dialog"))
except StopIteration as exc:
    cancelled_result = exc.value
else:
    raise AssertionError("cancellation should finish the wizard")

assert cancelled_result == WizardResult(
    sections={"account": {"username": "bob"}},
    cancelled=True,
    reason="user closed the dialog",
)


# Additional challenge problems with compact solutions

The following exercises add more code patterns while reusing the same delegation ideas.


## Challenge 13 — Delegate to a plain iterable and capture a default result

A normal list iterator has no meaningful return value. Create a delegator that yields all values from any iterable and returns the number of yielded values. Because `yield from iterable` cannot count by itself, wrap the iterable in a counting subgenerator.


In [29]:
def counting_iterator(values: Iterable[T]) -> Generator[T, None, int]:
    count = 0
    for value in values:
        count += 1
        yield value
    return count


def counted_delegation(values: Iterable[T]) -> Generator[T, None, int]:
    count = yield from counting_iterator(values)
    return count


items, item_count = drain(counted_delegation(["a", "b", "c", "d"]))
assert items == ["a", "b", "c", "d"]
assert item_count == 4


## Challenge 14 — Merge sorted streams and delegate the remaining tail

Merge two sorted iterables lazily. Once one iterator is exhausted, use `yield from` to emit the already-fetched current value and then the remainder of the other iterator.


In [30]:
def _prepend(first: T, remainder: Iterator[T]) -> Generator[T, None, None]:
    yield first
    yield from remainder


def merge_sorted(left: Iterable[T], right: Iterable[T]) -> Generator[T, None, None]:
    left_iter = iter(left)
    right_iter = iter(right)

    try:
        left_value = next(left_iter)
    except StopIteration:
        yield from right_iter
        return

    try:
        right_value = next(right_iter)
    except StopIteration:
        yield from _prepend(left_value, left_iter)
        return

    while True:
        if left_value <= right_value:
            yield left_value
            try:
                left_value = next(left_iter)
            except StopIteration:
                yield from _prepend(right_value, right_iter)
                return
        else:
            yield right_value
            try:
                right_value = next(right_iter)
            except StopIteration:
                yield from _prepend(left_value, left_iter)
                return


assert list(merge_sorted([1, 4, 7, 10], [2, 3, 8, 11])) == [
    1, 2, 3, 4, 7, 8, 10, 11
]
assert list(merge_sorted([], [1, 2])) == [1, 2]
assert list(merge_sorted([1, 2], [])) == [1, 2]


## Challenge 15 — A cleanup-safe nested context stream

Create several resource subgenerators and delegate to them sequentially. Closing the outer generator during the second resource must close the active second resource but must not open the third.


In [31]:
def finite_resource(
    name: str,
    values: Iterable[T],
    log: list[str],
) -> Generator[T, None, str]:
    log.append(f"open:{name}")
    try:
        yield from values
        return f"done:{name}"
    finally:
        log.append(f"close:{name}")


def resource_sequence(log: list[str]) -> Generator[int, None, list[str]]:
    reports: list[str] = []
    reports.append((yield from finite_resource("first", [1, 2], log)))
    reports.append((yield from finite_resource("second", [3, 4, 5], log)))
    reports.append((yield from finite_resource("third", [6], log)))
    return reports


resource_log: list[str] = []
sequence = resource_sequence(resource_log)
assert next(sequence) == 1
assert next(sequence) == 2
assert next(sequence) == 3  # first closed; second opened
sequence.close()

assert resource_log == [
    "open:first",
    "close:first",
    "open:second",
    "close:second",
]
assert "open:third" not in resource_log


# Concept summary

`yield from` is not merely shorthand for a `for` loop. It establishes a transparent delegation channel:

```text
caller -- next/send/throw/close --> delegator --> subgenerator
caller <-- yielded values -------- delegator <-- subgenerator
```

When the subgenerator returns, its return expression becomes `StopIteration.value`, and the `yield from` expression evaluates to that value inside the delegator.

The most important design consequence is separation of concerns: subgenerators can own local protocols and cleanup, while delegators compose those protocols with very little routing code.


# Final self-check

You should now be able to explain and demonstrate all of the following:

1. Why `yield from` is more powerful than a manual `for` loop.
2. How a subgenerator's return value reaches its delegator.
3. Why a generator must be primed before receiving a non-`None` sent value.
4. How `.throw(...)` reaches the active subgenerator.
5. Why cleanup belongs in `finally`.
6. How `.close()` propagates through nested delegation.
7. How recursive algorithms become concise with `yield from`.
8. How sequential subgenerator sessions can form a larger state machine.
9. Which parts of generator introspection are appropriate mainly for diagnostics.
10. Why native `yield from` is safer than hand-written protocol forwarding.
